# Negation Detection Analysis

Comprehensive analysis comparing:
1. **Sentiment-trained probes** (from 07_sweep_*.ipynb) - trained on SST-2 for sentiment classification
2. **Negation detection probes** (from 07_sweep_*_negation_detection.ipynb) - trained on JinaAI for negation detection

## Key Questions
- Which layer is best at detecting negation?
- Does Layer 3 show highest negation detection accuracy? (connects with cosine similarity findings)
- How do different pooling strategies compare for negation detection?

## Metrics Used (Only Calculated Metrics)
- `test_acc`: Test accuracy from sweep results
- `test_auroc`: Test AUROC from sweep results

Run this notebook **after** completing all sweep notebooks.


## 1. Setup


In [ ]:
# Mount Google Drive (for Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_DIR = '/content/drive/MyDrive/NOT_results'
except ImportError:
    # Local environment
    RESULTS_DIR = '../experiments'
    print("Running locally, using local experiments directory")

import os
print(f"Results directory: {RESULTS_DIR}")


In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")


## 2. Load Results


In [ ]:
def load_sweep_results(results_dir, sweep_name, results_file):
    """Load sweep results from JSON file."""
    path = os.path.join(results_dir, sweep_name, results_file)
    if os.path.exists(path):
        with open(path, 'r') as f:
            return json.load(f)
    else:
        print(f"Warning: {path} not found")
        return None

# Define result file paths
sentiment_sweeps = {
    'cls': ('sweep_cls', 'results_cls.json'),
    'mean': ('sweep_mean', 'results_mean.json'),
    'token': ('sweep_token', 'results_token.json'),
}

negation_sweeps = {
    'cls': ('sweep_cls_negation_detection', 'results_cls_negation_detection.json'),
    'mean': ('sweep_mean_negation_detection', 'results_mean_negation_detection.json'),
    'token': ('sweep_token_negation_detection', 'results_token_negation_detection.json'),
}

# Load all results
sentiment_results = {}
negation_results = {}

print("Loading Sentiment Sweep Results:")
print("=" * 50)
for pooling, (sweep_dir, results_file) in sentiment_sweeps.items():
    results = load_sweep_results(RESULTS_DIR, sweep_dir, results_file)
    if results:
        sentiment_results[pooling] = results
        print(f"  {pooling.upper()}: {len(results)} experiments loaded")
    else:
        print(f"  {pooling.upper()}: Not found")

print("\nLoading Negation Detection Sweep Results:")
print("=" * 50)
for pooling, (sweep_dir, results_file) in negation_sweeps.items():
    results = load_sweep_results(RESULTS_DIR, sweep_dir, results_file)
    if results:
        negation_results[pooling] = results
        print(f"  {pooling.upper()}: {len(results)} experiments loaded")
    else:
        print(f"  {pooling.upper()}: Not found")


## 3. Negation Detection Results Summary


In [ ]:
# Create summary DataFrame for negation detection results
if negation_results:
    all_negation_data = []
    for pooling, results in negation_results.items():
        for r in results:
            all_negation_data.append({
                'layer': r['layer'],
                'pooling': pooling.upper(),
                'test_acc': r.get('test_acc', 0),
                'test_auroc': r.get('test_auroc', 0),
            })
    
    negation_df = pd.DataFrame(all_negation_data)
    
    print("Negation Detection Results (All Pooling Strategies)")
    print("=" * 70)
    print("\nBy Layer and Pooling (sorted by AUROC):")
    print(negation_df.sort_values('test_auroc', ascending=False).to_string(index=False))
    
    # Find best overall
    best_idx = negation_df['test_auroc'].idxmax()
    best = negation_df.loc[best_idx]
    print(f"\n🏆 BEST OVERALL: Layer {int(best['layer'])} with {best['pooling']} pooling")
    print(f"   AUROC: {best['test_auroc']:.4f}, Accuracy: {best['test_acc']:.4f}")
    
    # Check if Layer 3 is best
    if best['layer'] == 3:
        print("\n✅ Layer 3 is best for negation detection!")
        print("   This CONNECTS with the cosine similarity findings.")
    else:
        print(f"\n⚠️ Layer {int(best['layer'])} is best (not Layer 3)")
else:
    print("No negation detection results found. Run the 07_sweep_*_negation_detection notebooks first.")


## 4. Layer-wise Comparison: Negation Detection


# Plot negation detection results by layer
if negation_results:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Prepare data for plotting
    layers = list(range(6))
    pooling_types = ['CLS', 'MEAN', 'TOKEN']
    colors = {'CLS': '#1f77b4', 'MEAN': '#ff7f0e', 'TOKEN': '#2ca02c'}
    
    # AUROC plot
    ax1 = axes[0]
    width = 0.25
    x = np.arange(len(layers))
    
    for i, pooling in enumerate(pooling_types):
        if pooling.lower() in negation_results:
            results = negation_results[pooling.lower()]
            results_sorted = sorted(results, key=lambda r: r['layer'])
            aurocs = [r.get('test_auroc', 0) for r in results_sorted]
            ax1.bar(x + i*width, aurocs, width, label=pooling, color=colors[pooling], alpha=0.8)
    
    ax1.set_xlabel('Layer')
    ax1.set_ylabel('Test AUROC')
    ax1.set_title('Negation Detection: AUROC by Layer and Pooling Strategy')
    ax1.set_xticks(x + width)
    ax1.set_xticklabels(layers)
    ax1.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Random baseline')
    ax1.legend()
    ax1.set_ylim(0, 1)
    
    # Accuracy plot
    ax2 = axes[1]
    for i, pooling in enumerate(pooling_types):
        if pooling.lower() in negation_results:
            results = negation_results[pooling.lower()]
            results_sorted = sorted(results, key=lambda r: r['layer'])
            accs = [r.get('test_acc', 0) for r in results_sorted]
            ax2.bar(x + i*width, accs, width, label=pooling, color=colors[pooling], alpha=0.8)
    
    ax2.set_xlabel('Layer')
    ax2.set_ylabel('Test Accuracy')
    ax2.set_title('Negation Detection: Accuracy by Layer and Pooling Strategy')
    ax2.set_xticks(x + width)
    ax2.set_xticklabels(layers)
    ax2.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Random baseline')
    ax2.legend()
    ax2.set_ylim(0, 1)
    
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'negation_detection_layer_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No negation detection results to plot")


In [ ]:
# Find best layer for each pooling strategy
if negation_results:
    print("Best Layer per Pooling Strategy (Negation Detection)")
    print("=" * 60)
    
    best_per_pooling = []
    for pooling, results in negation_results.items():
        if results:
            best = max(results, key=lambda r: r.get('test_auroc', 0))
            best_per_pooling.append({
                'Pooling': pooling.upper(),
                'Best Layer': best['layer'],
                'AUROC': best.get('test_auroc', 0),
                'Accuracy': best.get('test_acc', 0),
            })
            print(f"  {pooling.upper()}: Layer {best['layer']} (AUROC={best.get('test_auroc', 0):.4f})")
    
    # Check if all agree on Layer 3
    best_layers = [b['Best Layer'] for b in best_per_pooling]
    if all(l == 3 for l in best_layers):
        print("\n✅ ALL pooling strategies agree: Layer 3 is best!")
    elif 3 in best_layers:
        print(f"\n⚠️ Layer 3 is best for some pooling strategies")
    else:
        print(f"\n❌ Layer 3 is NOT best for any pooling strategy")
        print(f"   Best layers: {set(best_layers)}")
else:
    print("No results available")


In [ ]:
# Compare sentiment-trained vs negation detection probes
if sentiment_results and negation_results:
    print("Comparison: Sentiment-Trained vs Negation Detection Probes")
    print("=" * 70)
    
    comparison_data = []
    
    for pooling in ['cls', 'mean', 'token']:
        if pooling in sentiment_results and pooling in negation_results:
            sent_results = sentiment_results[pooling]
            neg_results = negation_results[pooling]
            
            # Best layer for each task
            sent_best = max(sent_results, key=lambda r: r.get('test_auroc', 0))
            neg_best = max(neg_results, key=lambda r: r.get('test_auroc', 0))
            
            comparison_data.append({
                'Pooling': pooling.upper(),
                'Sentiment Best Layer': sent_best['layer'],
                'Sentiment AUROC': sent_best.get('test_auroc', 0),
                'Negation Best Layer': neg_best['layer'],
                'Negation AUROC': neg_best.get('test_auroc', 0),
            })
    
    if comparison_data:
        comp_df = pd.DataFrame(comparison_data)
        print("\n" + comp_df.to_string(index=False))
        
        # Summary
        print("\n" + "-" * 70)
        print("Key Insight:")
        neg_best_layers = comp_df['Negation Best Layer'].tolist()
        if all(l == 3 for l in neg_best_layers):
            print("  ✅ Negation detection probes consistently find Layer 3 as best!")
            print("  This strongly connects with cosine similarity findings.")
        elif 3 in neg_best_layers:
            print(f"  ⚠️ Layer 3 is best for negation in some pooling strategies")
        else:
            print(f"  ❌ Negation detection does not favor Layer 3")
            print(f"     Best layers for negation: {set(neg_best_layers)}")
else:
    print("Need both sentiment and negation results for comparison")
    if not sentiment_results:
        print("  Missing: Sentiment sweep results")
    if not negation_results:
        print("  Missing: Negation detection sweep results")


In [ ]:
# Generate summary report
print("=" * 70)
print("NEGATION DETECTION ANALYSIS SUMMARY")
print("=" * 70)

if negation_results:
    # Find overall best
    all_results = []
    for pooling, results in negation_results.items():
        for r in results:
            all_results.append({
                'layer': r['layer'],
                'pooling': pooling,
                'auroc': r.get('test_auroc', 0),
                'acc': r.get('test_acc', 0),
            })
    
    best = max(all_results, key=lambda x: x['auroc'])
    
    print(f"\n1. BEST NEGATION DETECTION CONFIGURATION:")
    print(f"   Layer: {best['layer']}")
    print(f"   Pooling: {best['pooling'].upper()}")
    print(f"   AUROC: {best['auroc']:.4f}")
    print(f"   Accuracy: {best['acc']:.4f}")
    
    print(f"\n2. KEY FINDING:")
    if best['layer'] == 3:
        print("   ✅ Layer 3 IS BEST for negation detection!")
        print("   This CONNECTS with cosine similarity findings.")
        print("")
        print("   INTERPRETATION:")
        print("   - Distillation successfully transfers negation encoding to Layer 3")
        print("   - Layer 3 has both the STRUCTURE (cosine similarity) and")
        print("     the FUNCTIONAL CAPABILITY (probe accuracy) for negation")
    else:
        print(f"   ⚠️ Layer {best['layer']} is best (not Layer 3)")
        print("   This may indicate:")
        print("   - Cosine similarity pattern ≠ functional capability")
        print("   - Representation changes ≠ usable understanding")
    
    print(f"\n3. LAYER-WISE PERFORMANCE (Best pooling per layer):")
    for layer in range(6):
        layer_results = [r for r in all_results if r['layer'] == layer]
        if layer_results:
            best_for_layer = max(layer_results, key=lambda x: x['auroc'])
            marker = "🏆" if layer == best['layer'] else "  "
            print(f"   {marker} Layer {layer}: AUROC={best_for_layer['auroc']:.4f} ({best_for_layer['pooling'].upper()})")
    
    print("\n" + "=" * 70)
    print("END OF ANALYSIS")
    print("=" * 70)
else:
    print("\nNo negation detection results available.")
    print("Run the 07_sweep_*_negation_detection.ipynb notebooks first.")
